# Parent 정규화 단일 변수 대조

기존 VGGFace2 Parent는 비교 기준으로 보존하고, 같은 조건에서 Parent의 `weight_decay`만 0으로 바꿉니다.
새 초기화 → 첫 2 epoch 진단 → 유망할 때 같은 run을 40 epoch까지 재개 → SurvFace MagFace 1개 대조 순서입니다.
정규화 변경은 진단 조건이며 UCFace 저자의 공식 설정으로 표시하지 않습니다.


In [ ]:
from pathlib import Path
import sys
BASE_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fine-tune/modules").is_dir())
for p in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from modules.notebook_workflow import resolve_settings
from modules.parent_controls import make_parent_control, inspect_parent_control, launch_parent_control, GUIDE

# 기존 00 설정 파일을 읽기만 합니다. 공유 설정/기존 모델에 덮어쓰지 않습니다.
SETTINGS_PATH = Path("F:/face_datasets/.state/facex/shared_v1_boundary/notebook_settings_mobilefacenet.json")
REFERENCE_PARENT = Path("F:/models/facex/checkpoints/shared_v1_boundary_mobilefacenet/pretrain_arcface_seed8972/pretrain_mobilefacenet_1024d.pt")
REFERENCE_SHA256 = "3ac633038bed2bf153e2fe58aae900637881aba8c78c979d22712cc995cf879a"
OUTPUT_ROOT = Path("F:/models/facex/diagnostics/parent_weight_decay_zero_v1")
DATA_INTEGRITY_MODE = "immutable_verified_reuse"
FORCE_DATA_REVERIFY = False  # True일 때만 공유 전처리 이미지를 전체 재검증
EXECUTE = False  # True이면 새 Parent의 실제 학습/재개를 실행합니다.
STOP_AFTER_EPOCH = 2  # 총 일정은 40 epoch 유지. None=40까지; 2=epoch 2 저장 후 정지.
RESUME_CHECKPOINT = None  # 최초 실행 None. 재개할 때는 새 OUTPUT_ROOT 안의 checkpoint 전체 경로.
RUN_FIXED_VALIDATION = True  # EXECUTE=True의 학습 반환 후 고정 VGGFace2 표본 진단
KEEP_RAW_RESULTS = True  # epoch별 checkpoint 보존. False면 이전 중간 checkpoint 추적이 제한됩니다.

baseline = resolve_settings(BASE_DIR, SETTINGS_PATH, require_saved=True)
settings = make_parent_control(baseline, REFERENCE_PARENT, REFERENCE_SHA256, OUTPUT_ROOT,
    stop_after_epoch=STOP_AFTER_EPOCH, keep_raw_results=KEEP_RAW_RESULTS)
settings["data_integrity"] = dict(mode=DATA_INTEGRITY_MODE, force_reverify=FORCE_DATA_REVERIFY)
print("해석 안내:", GUIDE)


## 실행 전 비교

기존 Parent checkpoint와 실효 설정을 비교해 Parent weight_decay 이외의 과학적 차이가 있으면 차단합니다.
검증한 전처리 데이터의 불변 재사용 receipt를 확인합니다. 전체 이미지를 재전처리하지 않습니다.


In [ ]:
plan = inspect_parent_control(settings)
print({k: plan[k] for k in ("status", "run_id", "scientific_changes", "planned_epochs", "stop_after_epoch", "output_root")})
print("검증 재사용:", plan["data_verification"])
print("체크포인트 예상 GB:", round(plan["estimated_checkpoint_bytes"] / 1024**3, 2),
      "가용 GB:", round(plan["free_bytes"] / 1024**3, 2))


## Parent 실행 또는 재개

실행 중 Interrupt는 자식 프로세스에 안전 중단을 요청합니다. 출력되는 checkpoint/PID를 확인합니다.
2 epoch 진단 후 계속할 때는 새 checkpoint를 RESUME_CHECKPOINT에 넣고 STOP_AFTER_EPOCH=None으로 변경합니다.
기존 Parent 최종/복구 checkpoint로 새 정규화 실험을 재개할 수 없습니다.


In [ ]:
result = None
if EXECUTE:
    result = launch_parent_control(BASE_DIR, settings, resume_checkpoint=RESUME_CHECKPOINT)
    print(result)
else:
    print("계획 확인 완료. 학습은 시작하지 않았습니다.")


## 해석과 다음 단계

`summaries/<명시적 시각_ID>/summary.json`은 동일 epoch의 기존/신규 BN scale·분산과 학습 step을 비교합니다.
성분 수는 인식률이 아닙니다. 별도 고정 VGGFace2 test 진단 결과를 함께 확인해야 합니다.
2 epoch checkpoint는 미완료이며 후속 FR 학습에 사용할 수 없습니다.
40 epoch 완료 뒤 `make_followup_selection`으로 새 Parent 경로/해시를 등록합니다.
선택은 SurvFace 5,319명 resize_only MagFace seed 8972 한 개이고 fine-tune weight_decay는 기존 0.0001을 유지합니다.
후속 export와 ronbun Origin 평가에서 해당 selection/inventory를 명시적으로 사용합니다.


## 고정 진단 표본 평가
인물 단위 paired bootstrap으로 같은 epoch의 Rank-1 차이를 요약합니다. SurvFace 최종 test는 사용하지 않습니다.


In [ ]:
if EXECUTE and RUN_FIXED_VALIDATION:
    from modules.parent_validation import prepare_validation, evaluate_parent, compare_parents
    validation_root = OUTPUT_ROOT / "validation/vggface2_fixed_200x10_v1"
    prepare_validation(BASE_DIR, Path("F:/face_datasets/VGGFace2"), validation_root)
    reference_run = REFERENCE_PARENT.parent
    control_run = OUTPUT_ROOT / "checkpoint/pretrain_arcface_seed8972"
    for epoch in (1, 2, 40):
        name = f"pretrain_mobilefacenet_1024d_epoch_{epoch}.pt"
        if not (control_run/name).is_file():
            continue
        old = evaluate_parent(reference_run/name, validation_root, OUTPUT_ROOT/"validation/evaluations", keep_raw_results=KEEP_RAW_RESULTS)
        new = evaluate_parent(control_run/name, validation_root, OUTPUT_ROOT/"validation/evaluations", keep_raw_results=KEEP_RAW_RESULTS)
        comparison = OUTPUT_ROOT/"validation/comparisons"/f"epoch_{epoch}_{new.parent.name}.json"
        compare_parents(old, new, comparison)
        print("고정 표본 대조:", comparison)


## 완료 Parent의 후속 모델 연결
40 epoch 완료 후에만 SurvFace MagFace 하나의 선택 파일을 생성합니다. 학습 실행은 03, export는 04에서 같은 파일을 사용합니다.


In [ ]:
if EXECUTE:
    from modules.parent_controls import make_followup_selection
    from modules.checkpoint_manager import compute_sha256
    completed_parent = OUTPUT_ROOT / "checkpoint/pretrain_arcface_seed8972/pretrain_mobilefacenet_1024d.pt"
    if completed_parent.is_file():
        selection_path = make_followup_selection(baseline, completed_parent, compute_sha256(str(completed_parent)), OUTPUT_ROOT)
        print("03과 04에서 CAMPAIGN=parent_control로 사용할 선택 파일:", selection_path)
    else:
        print("40 epoch Parent 완료 전: 후속 모델 선택 파일은 아직 생성하지 않습니다.")
